# Ingesta de datos con Scala y Spark

## Portafolio Data Engineer

_____

### En esta seccion voy a trabajar la ingesta de distintos datos en varios formatos

El objetivo de este notebook es mostras las habilidades para ingestar distrintos tipos de archivos


In [1]:
import org.apache.spark.sql.SparkSession

// Inicialización de SparkSession (El punto de entrada a Spark)
val spark = SparkSession.builder()
  .appName("04_scala_spark")
  .master("local[*]") // Ejecutar localmente usando todos los cores disponibles
  //.master("spark://spark-master:7077") // Si activas este modo obtendrás algunos errores por la integración de Ammonite y Spark
  // Memoria del Driver (donde se recolectan los resultados de .collect())
  .config("spark.driver.memory", "2g") 
  // Memoria de cada Executor
  .config("spark.executor.memory", "2g")
  // Memoria adicional por encima del heap (útil para evitar errores de overhead)
  .config("spark.executor.memoryOverhead", "512m")
  //.config("deploy-mode","client") 
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") // Reducir el ruido en los logs

println(s"Spark Version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/17 02:47:20 INFO SparkContext: Running Spark version 4.1.1
26/02/17 02:47:20 INFO SparkContext: OS info Linux, 6.12.54-linuxkit, aarch64
26/02/17 02:47:20 INFO SparkContext: Java version 17.0.18+8-Ubuntu-122.04.1
26/02/17 02:47:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/17 02:47:20 INFO ResourceUtils: ==============================================================
26/02/17 02:47:20 INFO ResourceUtils: No custom resources configured for spark.driver.
26/02/17 02:47:20 INFO ResourceUtils: ==============================================================
26/02/17 02:47:20 INFO SparkContext: Submitted application: 04_scala_spark
26/02/17 02:47:20 INFO SecurityManager: Changing view acls to: jovyan
26/02/17 02:47:20 INFO SecurityManager: Changing modify acls to: jovyan
26/02/17 02:47:20 INFO SecurityManager: Changing vi

Spark Version: 4.1.1


import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@16e90907

### Uso de RDD

In [6]:
val csvDf = spark.read
  .option("header", "true")  
  .option("inferSchema", "true")
  .options(Map("delimiter" -> ","))
  .csv("/home/jovyan/data/product_stock_sample.csv")

csvDf.show(5)
csvDf.printSchema()

val rdd = csvDf.rdd


+---------+---------------+-------------+-----+----------+
|ProductID|    ProductName|ProductNumber|Color|TotalStock|
+---------+---------------+-------------+-----+----------+
|        1|Adjustable Race|      AR-5381| NULL|      1085|
|        3|BB Ball Bearing|      BE-2349| NULL|      1352|
|        2|   Bearing Ball|      BA-8327| NULL|      1109|
|      316|          Blade|      BL-2036| NULL|      1361|
|      324|    Chain Stays|      CS-2812| NULL|      1629|
+---------+---------------+-------------+-----+----------+
only showing top 5 rows
root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- TotalStock: integer (nullable = true)



csvDf: org.apache.spark.sql.package.DataFrame = [ProductID: int, ProductName: string ... 3 more fields]
rdd: org.apache.spark.rdd.RDD[org.apache.spark.sql.Row] = MapPartitionsRDD[35] at rdd at cmd6.sc:10

In [7]:
val maxStock = rdd.map(row => row.getAs[Int]("TotalStock")).max()
println(s"Máximo stock: $maxStock")

println()

val sumaTotalStock = rdd.map(row => row.getAs[Int]("TotalStock")).reduce(_ + _)
println(s"\nSuma total de stock: $sumaTotalStock")

Máximo stock: 1911


Suma total de stock: 114140


maxStock: Int = 1911
sumaTotalStock: Int = 114140

In [8]:
val productosAltoStock = rdd.filter(row => row.getAs[Int]("TotalStock") > 1500)
println("Productos con stock > 1500:")

productosAltoStock
.map(row => s"${row.getAs[String]("ProductName")}: ${row.getAs[Int]("TotalStock")}")
.take(5)
.foreach(println)


Productos con stock > 1500:
Chain Stays: 1629
Chainring: 1684
Chainring Nut: 1750
Crown Race: 1684
Decal 1: 1750


productosAltoStock: org.apache.spark.rdd.RDD[org.apache.spark.sql.Row] = MapPartitionsRDD[38] at filter at cmd8.sc:1

### Uso de DataFrame

In [2]:
import spark.implicits._ 

val jsonDf = spark.read
.option("inferSchema", "true")
.options(Map("multiline" -> "true"))
.json("/home/jovyan/data/product_sale_sample.json")

jsonDf.show(5)


+---------+--------------------+-------------------+--------------+
|ProductID|        product_name|total_quantity_sold| total_revenue|
+---------+--------------------+-------------------+--------------+
|      782|Mountain-200 Blac...|               2977|  4400592.8004|
|      783|Mountain-200 Blac...|               2664|4009494.761841|
|      779|Mountain-200 Silv...|               2394|3693678.025272|
|      780|Mountain-200 Silv...|               2234|3438478.860423|
|      781|Mountain-200 Silv...|               2216|3434256.941928|
+---------+--------------------+-------------------+--------------+
only showing top 5 rows


import spark.implicits._
jsonDf: org.apache.spark.sql.package.DataFrame = [ProductID: bigint, product_name: string ... 2 more fields]

In [ ]:
val filteredDF = jsonDf
.filter(col("total_revenue") > 1000)

filteredDF.show(5)

+---------+--------------------+-------------------+--------------+
|ProductID|        product_name|total_quantity_sold| total_revenue|
+---------+--------------------+-------------------+--------------+
|      782|Mountain-200 Blac...|               2977|  4400592.8004|
|      783|Mountain-200 Blac...|               2664|4009494.761841|
|      779|Mountain-200 Silv...|               2394|3693678.025272|
|      780|Mountain-200 Silv...|               2234|3438478.860423|
|      781|Mountain-200 Silv...|               2216|3434256.941928|
+---------+--------------------+-------------------+--------------+
only showing top 5 rows


filteredDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ProductID: bigint, product_name: string ... 2 more fields]

In [ ]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

//ranking de productos por total_revenue

val windowSpec = Window.orderBy(desc("total_revenue"))

val rankedDF = jsonDf
  .withColumn("rank", rank().over(windowSpec))

rankedDF.show()

+---------+--------------------+-------------------+--------------+----+
|ProductID|        product_name|total_quantity_sold| total_revenue|rank|
+---------+--------------------+-------------------+--------------+----+
|      782|Mountain-200 Blac...|               2977|  4400592.8004|   1|
|      783|Mountain-200 Blac...|               2664|4009494.761841|   2|
|      779|Mountain-200 Silv...|               2394|3693678.025272|   3|
|      780|Mountain-200 Silv...|               2234|3438478.860423|   4|
|      781|Mountain-200 Silv...|               2216|3434256.941928|   5|
|      784|Mountain-200 Blac...|               2111|3309673.216908|   6|
|      793|  Road-250 Black, 44|               1642|2516857.314918|   7|
|      794|  Road-250 Black, 48|               1498|2347655.953454|   8|
|      795|  Road-250 Black, 52|               1245|   2012447.775|   9|
|      753|    Road-150 Red, 56|                664|   1847818.628|  10|
|      976|Road-350-W Yellow...|               1622

import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
windowSpec: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@8ab16d2
rankedDF: org.apache.spark.sql.package.DataFrame = [ProductID: bigint, product_name: string ... 3 more fields]

In [ ]:
// Obtener los 10 productos con mayor total_revenue
val top10DF = jsonDf
  .orderBy(desc("total_revenue"))
  .limit(10)

top10DF.show()

+---------+--------------------+-------------------+--------------+
|ProductID|        product_name|total_quantity_sold| total_revenue|
+---------+--------------------+-------------------+--------------+
|      782|Mountain-200 Blac...|               2977|  4400592.8004|
|      783|Mountain-200 Blac...|               2664|4009494.761841|
|      779|Mountain-200 Silv...|               2394|3693678.025272|
|      780|Mountain-200 Silv...|               2234|3438478.860423|
|      781|Mountain-200 Silv...|               2216|3434256.941928|
|      784|Mountain-200 Blac...|               2111|3309673.216908|
|      793|  Road-250 Black, 44|               1642|2516857.314918|
|      794|  Road-250 Black, 48|               1498|2347655.953454|
|      795|  Road-250 Black, 52|               1245|   2012447.775|
|      753|    Road-150 Red, 56|                664|   1847818.628|
+---------+--------------------+-------------------+--------------+



top10DF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ProductID: bigint, product_name: string ... 2 more fields]